<a href="https://colab.research.google.com/github/ivansuarez-AU/EGN321_Module8_Assignment8_1/blob/main/EGN321_Module8_Assignment8_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGN 321 — Module 8 Assignment 8.1
# Complete Instructor Demo — Handoff, Dashboards, AI Analysis, and Reporting

## The Silent Handoff Challenge: Documentation, Operation, Analysis, and Improvement

This notebook is designed for a **live instructor demonstration** of Assignment 8.1.

It uses a realistic **900-record cooling-loop dataset** and demonstrates the complete lifecycle:

- data loading and quality checks;
- deterministic validation;
- SQLite history;
- structured logs;
- health and alert states;
- interactive dashboards;
- operational and engineering analysis;
- Silent Handoff scoring;
- **real Ollama AI assistance**;
- AI-assisted documentation audit;
- AI-assisted anomaly investigation;
- AI-assisted handoff review;
- professional Word reporting;
- HTML dashboard export;
- CSV/JSON evidence export;
- final submission/evidence ZIP.

> **Course rule:** Python, SQLite, tests, and logs establish the evidence.  
> Ollama may analyze, challenge, summarize, and suggest improvements — but it does not replace deterministic evidence.


## End-to-End Demo Architecture

```text
900-Record Sensor Dataset
        ↓
Data Quality Review
        ↓
Validation / Refusal Logic
        ↓
SQLite History + Structured Logs
        ↓
Operational Analysis
        ↓
Interactive Dashboard
        ↓
Silent Handoff Evidence
        ↓
Ollama Documentation Audit
        ↓
Ollama Anomaly / Incident Review
        ↓
Ollama Handoff Improvement Review
        ↓
Professional Report + AI_LOG.md
        ↓
Final Evidence ZIP
```

This gives students a realistic example of how an engineering application becomes something another person can **operate, understand, diagnose, and trust**.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import json
import shutil
import zipfile
import subprocess
import sys
import os
import math
import random
from datetime import datetime, timezone

DEMO_ROOT = Path("/content/module8_assignment8_1_complete_demo")
DEMO_ROOT.mkdir(exist_ok=True)

EXPORT_DIR = DEMO_ROOT / "final_export"
EXPORT_DIR.mkdir(exist_ok=True)

print("Demo root:", DEMO_ROOT)


Demo root: /content/module8_assignment8_1_complete_demo


# Part 1 — Load or Generate the 900-Record Dataset

For class, you may upload the supplied CSV.

If the file is not uploaded, the notebook can generate the same type of realistic dataset automatically so the demo does not stop.


In [2]:
from google.colab import files

EXPECTED_FILE = "cooling_loop_sensor_history_900_records.csv"

print("Upload the supplied CSV now, or cancel/skip and use the generated fallback.")

try:
    uploaded = files.upload()
except Exception:
    uploaded = {}

DATA_FILE = DEMO_ROOT / EXPECTED_FILE

if EXPECTED_FILE in uploaded:
    DATA_FILE.write_bytes(uploaded[EXPECTED_FILE])
    print("✅ Using uploaded dataset:", EXPECTED_FILE)

else:
    print("No supplied CSV detected. Generating the 900-record instructor dataset...")

    random.seed(81)
    np.random.seed(81)

    n = 900
    start = pd.Timestamp("2026-10-19 08:00:00", tz="UTC")
    timestamps = []
    current = start

    for i in range(n):
        if i == 520:
            current += pd.Timedelta(seconds=120)
        else:
            current += pd.Timedelta(seconds=10)
        timestamps.append(current)

    rows = []

    for i in range(n):
        record_id = i + 1

        temp = 24.0 + 0.45 * math.sin(i / 28) + np.random.normal(0, 0.16)
        pressure = 210.0 + 4.5 * math.sin(i / 35) + np.random.normal(0, 1.8)
        flow = 82.0 + 3.2 * math.sin(i / 22) + np.random.normal(0, 1.4)

        source_status = "OK"
        event_label = "NORMAL"

        if 250 <= i <= 279:
            temp = 30.4 + 0.6 * math.sin(i / 4) + np.random.normal(0, 0.10)
            event_label = "HIGH_TEMPERATURE_EVENT"

        if 355 <= i <= 362:
            flow = np.nan
            source_status = "MISSING"
            event_label = "MISSING_FLOW"

        if i in [430, 431]:
            temp = 86.0 if i == 430 else -18.0
            source_status = "REJECTED"
            event_label = "IMPOSSIBLE_TEMPERATURE"

        if 610 <= i <= 614:
            pressure = 485.0 + np.random.normal(0, 2)
            source_status = "REJECTED"
            event_label = "PRESSURE_OUT_OF_RANGE"

        if 720 <= i <= 742:
            flow = 20.5 + np.random.normal(0, 0.7)
            event_label = "LOW_FLOW_EVENT"

        if 743 <= i <= 755:
            flow = 55 + (i - 743) * 2.2 + np.random.normal(0, 0.5)
            event_label = "RECOVERY"

        rows.append({
            "record_id": record_id,
            "timestamp": timestamps[i].isoformat(),
            "temperature_c": None if pd.isna(temp) else round(float(temp), 3),
            "pressure_kpa": None if pd.isna(pressure) else round(float(pressure), 3),
            "flow_lpm": None if pd.isna(flow) else round(float(flow), 3),
            "source_status": source_status,
            "event_label": event_label
        })

    pd.DataFrame(rows).to_csv(DATA_FILE, index=False)
    print("✅ Generated:", DATA_FILE)

sensor_df = pd.read_csv(DATA_FILE)
sensor_df["timestamp"] = pd.to_datetime(sensor_df["timestamp"], utc=True)

print("Records:", len(sensor_df))
print("Columns:", list(sensor_df.columns))
display(sensor_df.head())


Upload the supplied CSV now, or cancel/skip and use the generated fallback.


Saving cooling_loop_sensor_history_900_records.csv to cooling_loop_sensor_history_900_records.csv
✅ Using uploaded dataset: cooling_loop_sensor_history_900_records.csv
Records: 900
Columns: ['record_id', 'timestamp', 'temperature_c', 'pressure_kpa', 'flow_lpm', 'source_status', 'event_label']


,record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,source_status,event_label
0,1,2026-10-19 08:00:10+00:00,23.782,207.776,82.084,OK,NORMAL
1,2,2026-10-19 08:00:20+00:00,23.751,210.171,82.098,OK,NORMAL
2,3,2026-10-19 08:00:30+00:00,23.841,206.474,82.806,OK,NORMAL
3,4,2026-10-19 08:00:40+00:00,24.287,210.894,80.196,OK,NORMAL
4,5,2026-10-19 08:00:50+00:00,24.038,208.463,85.083,OK,NORMAL


# Part 2 — Data Quality and Engineering Reality Check

Before AI, dashboards, or reports, we establish deterministic facts about the data.

We will inspect:

- record count;
- missing values;
- source status;
- event distribution;
- sampling gaps;
- descriptive statistics;
- supported engineering ranges.


In [3]:
quality_summary = pd.DataFrame({
    "metric": [
        "Total Records",
        "Missing Temperature",
        "Missing Pressure",
        "Missing Flow",
        "Non-OK Source Status",
        "Unique Event Labels"
    ],
    "value": [
        len(sensor_df),
        int(sensor_df["temperature_c"].isna().sum()),
        int(sensor_df["pressure_kpa"].isna().sum()),
        int(sensor_df["flow_lpm"].isna().sum()),
        int((sensor_df["source_status"] != "OK").sum()),
        int(sensor_df["event_label"].nunique())
    ]
})

display(quality_summary)

print("Event distribution:")
display(
    sensor_df["event_label"]
    .value_counts()
    .rename_axis("event")
    .reset_index(name="count")
)

print("Source status distribution:")
display(
    sensor_df["source_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print("Descriptive statistics:")
display(
    sensor_df[
        ["temperature_c", "pressure_kpa", "flow_lpm"]
    ].describe().round(3)
)


,metric,value
0,Total Records,900
1,Missing Temperature,0
2,Missing Pressure,0
3,Missing Flow,8
4,Non-OK Source Status,15
5,Unique Event Labels,7


Event distribution:


,event,count
0,NORMAL,819
1,HIGH_TEMPERATURE_EVENT,30
2,LOW_FLOW_EVENT,23
3,RECOVERY,13
4,MISSING_FLOW,8
5,PRESSURE_OUT_OF_RANGE,5
6,IMPOSSIBLE_TEMPERATURE,2


Source status distribution:


,status,count
0,OK,885
1,MISSING,8
2,REJECTED,7


Descriptive statistics:


,temperature_c,pressure_kpa,flow_lpm
count,900.000,900.000,892.000
mean,24.238,211.625,80.278
std,2.774,20.809,10.247
min,-18.000,201.600,19.385
25%,23.689,207.272,79.783
50%,24.042,210.208,81.918
75%,24.322,213.026,84.125
max,86.000,487.728,90.733


In [4]:
sensor_df["gap_seconds"] = (
    sensor_df["timestamp"]
    .diff()
    .dt.total_seconds()
    .fillna(0)
)

gap_table = sensor_df[
    sensor_df["gap_seconds"] > 20
][
    ["record_id", "timestamp", "gap_seconds", "event_label"]
].copy()

print("Sampling gaps > 20 seconds:")
display(gap_table)


Sampling gaps > 20 seconds:


,record_id,timestamp,gap_seconds,event_label
520,521,2026-10-19 09:28:40+00:00,120.0,NORMAL


# Part 3 — Deterministic Validation, Health, Alerts, SQLite, and Logs

The application will distinguish:

- raw values;
- accepted values;
- rejected values;
- HEALTHY / DEGRADED state;
- HIGH_TEMPERATURE alert;
- LOW_FLOW alert;
- timing-gap evidence.

This is the source of truth used later by Ollama.


In [5]:
def validate_reading(temp_c, pressure_kpa, flow_lpm):
    reasons = []

    if pd.isna(temp_c):
        reasons.append("temperature missing")
    elif not 0 <= float(temp_c) <= 60:
        reasons.append(f"temperature out of range: {temp_c}")

    if pd.isna(pressure_kpa):
        reasons.append("pressure missing")
    elif not 50 <= float(pressure_kpa) <= 400:
        reasons.append(f"pressure out of range: {pressure_kpa}")

    if pd.isna(flow_lpm):
        reasons.append("flow missing")
    elif not 0 <= float(flow_lpm) <= 200:
        reasons.append(f"flow out of range: {flow_lpm}")

    return {
        "accepted": len(reasons) == 0,
        "reasons": reasons
    }


class AlertEngine:
    def __init__(
        self,
        temp_trigger=30.0,
        temp_clear=28.5,
        low_flow_trigger=25.0,
        low_flow_clear=35.0,
        consecutive=3
    ):
        self.temp_trigger = temp_trigger
        self.temp_clear = temp_clear
        self.low_flow_trigger = low_flow_trigger
        self.low_flow_clear = low_flow_clear
        self.consecutive = consecutive

        self.temp_count = 0
        self.flow_count = 0

        self.high_temp = False
        self.low_flow = False

    def update(self, temp_c, flow_lpm, accepted):
        if not accepted:
            self.temp_count = 0
            self.flow_count = 0
            return self.state()

        if not self.high_temp:
            self.temp_count = self.temp_count + 1 if temp_c >= self.temp_trigger else 0
            if self.temp_count >= self.consecutive:
                self.high_temp = True
        elif temp_c <= self.temp_clear:
            self.high_temp = False
            self.temp_count = 0

        if not self.low_flow:
            self.flow_count = self.flow_count + 1 if flow_lpm < self.low_flow_trigger else 0
            if self.flow_count >= self.consecutive:
                self.low_flow = True
        elif flow_lpm >= self.low_flow_clear:
            self.low_flow = False
            self.flow_count = 0

        return self.state()

    def state(self):
        active = []

        if self.high_temp:
            active.append("HIGH_TEMPERATURE")

        if self.low_flow:
            active.append("LOW_FLOW")

        return active or ["NORMAL"]


In [6]:
DB_FILE = DEMO_ROOT / "cooling_loop.db"
LOG_FILE = DEMO_ROOT / "system_log.jsonl"

for p in [DB_FILE, LOG_FILE]:
    if p.exists():
        p.unlink()

conn = sqlite3.connect(DB_FILE)

conn.execute('''
CREATE TABLE readings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    record_id INTEGER,
    timestamp TEXT,
    temperature_c REAL,
    pressure_kpa REAL,
    flow_lpm REAL,
    accepted INTEGER,
    health_state TEXT,
    alert_state TEXT,
    reason TEXT,
    gap_seconds REAL
)
''')

engine = AlertEngine()
previous_ts = None

for _, row in sensor_df.iterrows():

    ts = row["timestamp"]

    gap_seconds = (
        0
        if previous_ts is None
        else (ts - previous_ts).total_seconds()
    )

    previous_ts = ts

    result = validate_reading(
        row["temperature_c"],
        row["pressure_kpa"],
        row["flow_lpm"]
    )

    accepted = result["accepted"]
    reason = "; ".join(result["reasons"])

    health = "HEALTHY"

    if not accepted or gap_seconds > 30:
        health = "DEGRADED"

    temp = (
        None
        if pd.isna(row["temperature_c"])
        else float(row["temperature_c"])
    )

    pressure = (
        None
        if pd.isna(row["pressure_kpa"])
        else float(row["pressure_kpa"])
    )

    flow = (
        None
        if pd.isna(row["flow_lpm"])
        else float(row["flow_lpm"])
    )

    alerts = engine.update(
        temp,
        flow,
        accepted
    )

    conn.execute(
        '''INSERT INTO readings
        (record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,
         accepted,health_state,alert_state,reason,gap_seconds)
        VALUES(?,?,?,?,?,?,?,?,?,?)''',
        (
            int(row["record_id"]),
            ts.isoformat(),
            temp,
            pressure,
            flow,
            int(accepted),
            health,
            ",".join(alerts),
            reason,
            gap_seconds
        )
    )

    log_record = {
        "timestamp": ts.isoformat(),
        "record_id": int(row["record_id"]),
        "component": "ingestion",
        "event": "reading_processed",
        "accepted": bool(accepted),
        "health_state": health,
        "alert_state": alerts,
        "reason": reason,
        "gap_seconds": gap_seconds
    }

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(log_record) + "\n")

conn.commit()

history = pd.read_sql_query(
    "SELECT * FROM readings ORDER BY id",
    conn
)

conn.close()

print("Database rows:", len(history))
print("Log file:", LOG_FILE)


Database rows: 900
Log file: /content/module8_assignment8_1_complete_demo/system_log.jsonl


# Part 4 — Operational KPI Analysis

A professional dashboard should begin with a small number of useful operational KPIs.


In [7]:
total_records = len(history)
accepted_records = int((history["accepted"] == 1).sum())
rejected_records = int((history["accepted"] == 0).sum())
degraded_records = int((history["health_state"] == "DEGRADED").sum())

high_temp_rows = int(
    history["alert_state"]
    .str.contains("HIGH_TEMPERATURE", na=False)
    .sum()
)

low_flow_rows = int(
    history["alert_state"]
    .str.contains("LOW_FLOW", na=False)
    .sum()
)

large_gaps = int((history["gap_seconds"] > 30).sum())

acceptance_rate = accepted_records / total_records * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Records",
        "Accepted Records",
        "Rejected Records",
        "Acceptance Rate",
        "Degraded Records",
        "High Temperature Alert Rows",
        "Low Flow Alert Rows",
        "Large Sampling Gaps"
    ],
    "Value": [
        total_records,
        accepted_records,
        rejected_records,
        f"{acceptance_rate:.2f}%",
        degraded_records,
        high_temp_rows,
        low_flow_rows,
        large_gaps
    ]
})

display(kpis)


,KPI,Value
0,Total Records,900
1,Accepted Records,885
2,Rejected Records,15
3,Acceptance Rate,98.33%
4,Degraded Records,16
5,High Temperature Alert Rows,28
6,Low Flow Alert Rows,21
7,Large Sampling Gaps,1


# Part 5 — Interactive Operations Dashboard

This section creates real interactive Plotly dashboards.

The instructor can zoom into anomalies, hover over records, and show how the dashboard supports the handoff.


In [8]:
%pip install -q plotly

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
plot_df = history.copy()
plot_df["timestamp"] = pd.to_datetime(
    plot_df["timestamp"],
    utc=True
)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Temperature",
        "Pressure",
        "Flow"
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["temperature_c"],
        mode="lines",
        name="Temperature C"
    ),
    row=1,
    col=1
)

fig.add_hline(
    y=30,
    line_dash="dash",
    annotation_text="High-temp trigger",
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["pressure_kpa"],
        mode="lines",
        name="Pressure kPa"
    ),
    row=2,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["flow_lpm"],
        mode="lines",
        name="Flow L/min"
    ),
    row=3,
    col=1
)

fig.add_hline(
    y=25,
    line_dash="dash",
    annotation_text="Low-flow trigger",
    row=3,
    col=1
)

fig.update_layout(
    height=850,
    title="Cooling Loop Operations Dashboard — Time-Series View",
    hovermode="x unified"
)

fig.show()


In [9]:
status_fig = px.bar(
    history["health_state"]
    .value_counts()
    .rename_axis("health_state")
    .reset_index(name="count"),
    x="health_state",
    y="count",
    title="Health-State Distribution"
)

status_fig.show()

event_fig = px.bar(
    sensor_df["event_label"]
    .value_counts()
    .rename_axis("event_label")
    .reset_index(name="count"),
    x="event_label",
    y="count",
    title="Scenario / Event Distribution"
)

event_fig.update_xaxes(tickangle=35)
event_fig.show()


# Part 6 — Anomaly and Incident Analysis

Instead of only plotting the data, we extract operationally meaningful records:

- rejected readings;
- degraded health;
- high-temperature alerts;
- low-flow alerts;
- timing gaps;
- recovery records.


In [10]:
incident_mask = (
    (history["accepted"] == 0)
    | (history["health_state"] != "HEALTHY")
    | history["alert_state"].str.contains(
        "HIGH_TEMPERATURE|LOW_FLOW",
        regex=True,
        na=False
    )
    | (history["gap_seconds"] > 30)
)

incident_records = history.loc[
    incident_mask,
    [
        "record_id",
        "timestamp",
        "temperature_c",
        "pressure_kpa",
        "flow_lpm",
        "accepted",
        "health_state",
        "alert_state",
        "reason",
        "gap_seconds"
    ]
].copy()

print("Operationally significant rows:", len(incident_records))
display(incident_records.head(30))


Operationally significant rows: 65


,record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,accepted,health_state,alert_state,reason,gap_seconds
252,253,2026-10-19T08:42:10+00:00,30.375,215.027,78.902,1,HEALTHY,HIGH_TEMPERATURE,,10.0
253,254,2026-10-19T08:42:20+00:00,30.713,212.310,77.797,1,HEALTHY,HIGH_TEMPERATURE,,10.0
254,255,2026-10-19T08:42:30+00:00,30.898,215.941,79.728,1,HEALTHY,HIGH_TEMPERATURE,,10.0
255,256,2026-10-19T08:42:40+00:00,30.939,213.747,81.692,1,HEALTHY,HIGH_TEMPERATURE,,10.0
256,257,2026-10-19T08:42:50+00:00,31.091,210.578,80.605,1,HEALTHY,HIGH_TEMPERATURE,,10.0
257,258,2026-10-19T08:43:00+00:00,31.018,217.344,78.093,1,HEALTHY,HIGH_TEMPERATURE,,10.0
258,259,2026-10-19T08:43:10+00:00,31.014,214.445,79.190,1,HEALTHY,HIGH_TEMPERATURE,,10.0
259,260,2026-10-19T08:43:20+00:00,30.962,213.437,82.236,1,HEALTHY,HIGH_TEMPERATURE,,10.0
260,261,2026-10-19T08:43:30+00:00,30.868,212.521,78.261,1,HEALTHY,HIGH_TEMPERATURE,,10.0
261,262,2026-10-19T08:43:40+00:00,30.890,213.443,77.508,1,HEALTHY,HIGH_TEMPERATURE,,10.0


In [11]:
analysis_summary = {
    "total_records": int(total_records),
    "accepted_records": int(accepted_records),
    "rejected_records": int(rejected_records),
    "acceptance_rate_percent": round(float(acceptance_rate), 3),
    "degraded_records": int(degraded_records),
    "high_temperature_alert_rows": int(high_temp_rows),
    "low_flow_alert_rows": int(low_flow_rows),
    "large_sampling_gaps": int(large_gaps),
    "temperature_mean_c": round(float(history["temperature_c"].mean()), 3),
    "temperature_max_c": round(float(history["temperature_c"].max()), 3),
    "pressure_mean_kpa": round(float(history["pressure_kpa"].mean()), 3),
    "pressure_max_kpa": round(float(history["pressure_kpa"].max()), 3),
    "flow_mean_lpm": round(float(history["flow_lpm"].mean()), 3),
    "flow_min_lpm": round(float(history["flow_lpm"].min()), 3),
    "operationally_significant_rows": int(len(incident_records))
}

print(json.dumps(analysis_summary, indent=2))


{
  "total_records": 900,
  "accepted_records": 885,
  "rejected_records": 15,
  "acceptance_rate_percent": 98.333,
  "degraded_records": 16,
  "high_temperature_alert_rows": 28,
  "low_flow_alert_rows": 21,
  "large_sampling_gaps": 1,
  "temperature_mean_c": 24.238,
  "temperature_max_c": 86.0,
  "pressure_mean_kpa": 211.625,
  "pressure_max_kpa": 487.728,
  "flow_mean_lpm": 80.278,
  "flow_min_lpm": 19.385,
  "operationally_significant_rows": 65
}


# Part 7 — Handoff Dashboard and Operator Analysis

Assignment 8.1 is not only about the sensor data.

We also analyze the **handoff itself**:

- completion rate;
- partial completions;
- points of confusion;
- required improvements;
- final handoff level.


In [12]:
handoff_tasks = pd.DataFrame([
    [1, "Locate the application and repository files"],
    [2, "Explain what the system is designed to do"],
    [3, "Install or verify required dependencies"],
    [4, "Start the application"],
    [5, "Complete one normal workflow"],
    [6, "Provide or identify valid input"],
    [7, "Interpret the displayed result"],
    [8, "Locate health/status information"],
    [9, "Locate the logs"],
    [10, "Identify one documented failure condition"],
    [11, "Follow one troubleshooting/recovery procedure"],
    [12, "Identify one known limitation"]
], columns=["step", "handoff_task"])

handoff_observation = pd.DataFrame([
    [1,"Locate application","Yes","Found app.py and README quickly","None"],
    [2,"Understand purpose","Yes","Purpose clear","None"],
    [3,"Verify dependencies","Yes","requirements.txt matched README","None"],
    [4,"Start application","Yes","Startup command clear","None"],
    [5,"Normal workflow","Yes","Dashboard opened and showed 900 records","None"],
    [6,"Valid input","Yes","Included dataset clearly identified","None"],
    [7,"Interpret result","Partial","Alert meanings required rereading","Add alert examples"],
    [8,"Locate health/status","Yes","Found dashboard/API status","None"],
    [9,"Locate logs","Yes","Log path documented","None"],
    [10,"Failure condition","Yes","Database missing scenario documented","None"],
    [11,"Recovery procedure","Yes","Recovery worked but success check was not prominent","Make verification explicit"],
    [12,"Known limitation","Yes","Found limitations section","None"]
], columns=[
    "step",
    "operator_action",
    "success",
    "confusion_observed",
    "improvement_needed"
])

score_map = {
    "Yes": 1.0,
    "Partial": 0.5,
    "No": 0.0
}

handoff_observation["score"] = (
    handoff_observation["success"]
    .map(score_map)
)

handoff_score = float(
    handoff_observation["score"].sum()
)

handoff_max = len(handoff_observation)

handoff_percent = (
    handoff_score / handoff_max * 100
)

if handoff_percent >= 95:
    handoff_level = "Level 5 - Professional Handoff"
elif handoff_percent >= 85:
    handoff_level = "Level 4 - Transferable"
elif handoff_percent >= 70:
    handoff_level = "Level 3 - Operational"
elif handoff_percent >= 50:
    handoff_level = "Level 2 - Almost Operational"
else:
    handoff_level = "Level 1 - Needs the Developer"

print(
    f"Handoff score: {handoff_score:.1f}/{handoff_max} "
    f"({handoff_percent:.1f}%)"
)
print(handoff_level)

display(handoff_observation)


Handoff score: 11.5/12 (95.8%)
Level 5 - Professional Handoff


,step,operator_action,success,confusion_observed,improvement_needed,score
0,1,Locate application,Yes,Found app.py and README quickly,None,1.0
1,2,Understand purpose,Yes,Purpose clear,None,1.0
2,3,Verify dependencies,Yes,requirements.txt matched README,None,1.0
3,4,Start application,Yes,Startup command clear,None,1.0
4,5,Normal workflow,Yes,Dashboard opened and showed 900 records,None,1.0
5,6,Valid input,Yes,Included dataset clearly identified,None,1.0
6,7,Interpret result,Partial,Alert meanings required rereading,Add alert examples,0.5
7,8,Locate health/status,Yes,Found dashboard/API status,None,1.0
8,9,Locate logs,Yes,Log path documented,None,1.0
9,10,Failure condition,Yes,Database missing scenario documented,None,1.0


In [13]:
handoff_fig = px.bar(
    handoff_observation,
    x="step",
    y="score",
    hover_data=[
        "operator_action",
        "confusion_observed",
        "improvement_needed"
    ],
    title="Silent Handoff Task Completion"
)

handoff_fig.update_yaxes(
    range=[0, 1.1],
    title="Completion Score"
)

handoff_fig.show()


# Part 8 — Install and Run Ollama in Colab

This is the AI assistance that should be visible during the demo.

We will use:

**Ollama + qwen2.5:0.5b**

Ollama will be used in three different roles:

1. **Documentation Reviewer**
2. **Operational / Anomaly Investigator**
3. **Handoff Improvement Reviewer**

AI is not allowed to overwrite the deterministic results.


In [14]:
import subprocess
import shutil as _shutil
import time as _time
import requests

def shell(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout[-2500:])

    if result.returncode != 0 and result.stderr:
        print(result.stderr[-2500:])

    return result.returncode

shell("apt-get update -qq")
shell("apt-get install -y -qq curl zstd")

if _shutil.which("ollama") is None:
    shell("curl -fsSL https://ollama.com/install.sh | sh")

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "qwen2.5:0.5b"

def ollama_running():
    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=3
        )
        return response.status_code == 200
    except requests.RequestException:
        return False

if not ollama_running():
    log_handle = open(
        "/tmp/ollama_module8.log",
        "a"
    )

    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT
    )

    for _ in range(45):
        if ollama_running():
            break
        _time.sleep(1)

if not ollama_running():
    raise RuntimeError(
        "Ollama did not start successfully."
    )

shell(
    f"ollama pull {OLLAMA_MODEL}"
)

print("✅ Ollama ready:", OLLAMA_MODEL)


Selecting previously unselected package zstd.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


✅ Ollama ready: qwen2.5:0.5b


In [15]:
AI_SYSTEM_POLICY = '''
You are an engineering-software review assistant for EGN 321.

Use only the supplied evidence.

Rules:
1. Do not invent readings, timestamps, alerts, logs, files, test results, or application features.
2. Clearly separate observation, inference, hypothesis, and recommendation.
3. Deterministic Python/SQLite/log results are authoritative.
4. If evidence is insufficient, say what evidence is missing.
5. Do not claim a root cause is proven unless the evidence proves it.
6. For documentation reviews, act like a first-time operator.
7. For handoff reviews, focus on what prevented independent operation.
8. Be concise enough to use in a classroom report.
'''

def ask_ollama(
    prompt,
    timeout=600,
    num_predict=900
):
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "system": AI_SYSTEM_POLICY,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0.2,
            "num_predict": num_predict,
            "num_ctx": 4096
        }
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=(10, timeout)
    )

    response.raise_for_status()

    return (
        response
        .json()["response"]
        .strip()
    )

print(
    "Warm-up:",
    ask_ollama(
        "Reply with exactly READY.",
        num_predict=10
    )
)


Warm-up: READY


# Part 9 — Ollama Documentation Audit

Ollama reviews the documentation as if it were a first-time operator.

The goal is not to ask AI to write the documentation for the student.

The goal is to identify:

- missing prerequisites;
- vague startup instructions;
- hidden assumptions;
- weak success checks;
- missing recovery information;
- undocumented limitations.


In [16]:
README_SAMPLE = '''
# Cooling Loop Reliability Monitor

## Purpose
Monitor cooling-loop temperature, pressure, and flow.

## Requirements
Python 3.10+, Flask, pandas, pytest, browser, SQLite write access.

## Quick Start
1. Install requirements.
2. Build the database.
3. Start the Flask app.
4. Open http://127.0.0.1:5000.
5. Confirm 900 records appear.

## Inputs
Temperature: Celsius, 0-60 C.
Pressure: kPa, 50-400 kPa.
Flow: L/min, 0-200 L/min.

## Refusal
Missing or unsupported values are rejected.

## Logs
system_log.jsonl.

## Recovery
If the database is missing, rebuild it, restart the app,
and verify that the dashboard reports 900 records.

## Limitation
The classroom demo does not independently verify physical sensor calibration.
'''

documentation_prompt = f'''
Review the following operating guide as a first-time operator.

OPERATING GUIDE:
{README_SAMPLE}

Return these sections:
1. What is clear
2. Missing prerequisites
3. Ambiguous instructions
4. Hidden assumptions
5. Missing success checks
6. Missing troubleshooting/recovery details
7. Missing limitations
8. Top 5 documentation improvements

Do not invent application features.
'''

ollama_documentation_review = ask_ollama(
    documentation_prompt,
    num_predict=1000
)

print(ollama_documentation_review)


1. What is clear:
   - The operating guide is a first-time operator's manual for a software application.
   - It describes the purpose, requirements, and initial setup.
   - It outlines the expected outcomes and the steps to follow.
   - It provides a quick start guide and a list of inputs and outputs.

2. Missing prerequisites:
   - Python 3.10+
   - Flask
   - pandas
   - pytest
   - browser
   - SQLite write access

3. Ambiguous instructions:
   - The guide does not explicitly state what to do next after starting the app.
   - It mentions "confirm 900 records appear" but does not specify how to do so.

4. Hidden assumptions:
   - The guide assumes that the user has basic knowledge of the application and its functionalities.
   - It assumes that the user has the necessary permissions and access to the database.

5. Missing success checks:
   - The guide does not explicitly state what to do if the database is missing.
   - It mentions "confirm 900 records appear" but does not specify 

# Part 10 — Ollama Operational / Anomaly Review

Now Ollama receives only a deterministic summary and selected incident records.

This shows students a good AI pattern:

> **Python finds the evidence. AI helps interpret and communicate it.**


In [17]:
incident_for_ai = (
    incident_records
    .head(35)
    .fillna("")
    .to_dict(orient="records")
)

operational_prompt = f'''
Review this deterministic operational evidence.

SUMMARY:
{json.dumps(analysis_summary, indent=2)}

SELECTED INCIDENT RECORDS:
{json.dumps(incident_for_ai, indent=2, default=str)}

Return:
1. Executive operational summary
2. Most important observed conditions
3. What the evidence proves
4. What it does NOT prove
5. Three plausible engineering/operational hypotheses
6. Missing evidence needed to test each hypothesis
7. Recommended deterministic checks
8. What should appear in the operator handoff documentation

Do not invent events or causes.
'''

ollama_operational_review = ask_ollama(
    operational_prompt,
    num_predict=1200
)

print(ollama_operational_review)


1. Executive operational summary:
The data provided is a list of temperature readings taken at various times during a specific period. The data shows that the temperature readings are consistent and do not show any significant fluctuations. The data also shows that the flow readings are consistent and do not show any significant fluctuations. The data does not show any temperature or pressure anomalies.

2. Most important observed conditions:
The most important observed conditions are the consistent temperature readings and the consistent flow readings. These conditions are the primary indicators of the system's operational status.

3. What the evidence proves:
The evidence proves that the system is operating within its normal operating range. The data shows that the temperature readings are within the range of 24°C to 25°C, and the flow readings are within the range of 2000 to 2500 L/min.

4. What it does NOT prove:
The evidence does not prove that the system is operating within its n

# Part 11 — Ollama Silent Handoff Review

This use of AI is especially relevant to Assignment 8.1.

Ollama reviews the **human handoff evidence** and proposes documentation/application improvements.


In [ ]:
handoff_for_ai = (
    handoff_observation
    .drop(columns=["score"])
    .to_dict(orient="records")
)

handoff_prompt = f'''
Review this silent handoff evidence.

HANDOFF SCORE:
{handoff_score}/{handoff_max}
Percent: {handoff_percent:.1f}
Level: {handoff_level}

OBSERVATIONS:
{json.dumps(handoff_for_ai, indent=2)}

Return:
1. What the operator completed independently
2. Where the operator showed friction or uncertainty
3. Which issue is documentation-related
4. Which issue may require an application/interface improvement
5. At least two meaningful improvements
6. How each improvement should be verified in a second handoff
7. One risk the author may be overlooking

Do not invent operator behavior.
'''

ollama_handoff_review = ask_ollama(
    handoff_prompt,
    num_predict=1000
)

print(ollama_handoff_review)


# Part 12 — AI Red-Team Review

Before accepting the AI summary, ask a second AI pass to challenge overconfident conclusions.


In [ ]:
red_team_prompt = f'''
You are red-teaming the following AI operational review:

{ollama_operational_review}

Deterministic facts:
{json.dumps(analysis_summary, indent=2)}

Identify:
1. any claim that is stronger than the evidence;
2. any implied root cause that is not proven;
3. any missing caveat;
4. any recommendation that needs a deterministic verification test;
5. a safer revised conclusion.

Use only the evidence provided.
'''

ollama_red_team_review = ask_ollama(
    red_team_prompt,
    num_predict=900
)

print(ollama_red_team_review)


# Part 13 — Export a Standalone HTML Dashboard

The notebook creates an interactive HTML dashboard that can be opened outside Colab.

This is useful for:

- instructor demonstration;
- Assignment 8.1 handoff;
- capstone evidence;
- GitHub documentation.


In [ ]:
from plotly.offline import plot

DASHBOARD_HTML = EXPORT_DIR / "cooling_loop_dashboard.html"

dashboard = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=(
        "Temperature History",
        "Pressure History",
        "Flow History",
        "Handoff Completion"
    )
)

dashboard.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["temperature_c"],
        mode="lines",
        name="Temperature C"
    ),
    row=1,
    col=1
)

dashboard.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["pressure_kpa"],
        mode="lines",
        name="Pressure kPa"
    ),
    row=2,
    col=1
)

dashboard.add_trace(
    go.Scatter(
        x=plot_df["timestamp"],
        y=plot_df["flow_lpm"],
        mode="lines",
        name="Flow L/min"
    ),
    row=3,
    col=1
)

dashboard.add_trace(
    go.Bar(
        x=handoff_observation["step"],
        y=handoff_observation["score"],
        name="Handoff Score"
    ),
    row=4,
    col=1
)

dashboard.update_layout(
    height=1200,
    title=(
        "EGN 321 Module 8 Assignment 8.1 — "
        "Operations + Handoff Dashboard"
    )
)

plot(
    dashboard,
    filename=str(DASHBOARD_HTML),
    auto_open=False,
    include_plotlyjs=True
)

print("Created:", DASHBOARD_HTML)


# Part 14 — Create Static Figures for the Professional Report

The Word report will include the same evidence in a format that is easy to submit or review.


In [ ]:
import matplotlib.pyplot as plt

TEMP_PNG = EXPORT_DIR / "temperature_analysis.png"
FLOW_PNG = EXPORT_DIR / "flow_analysis.png"
HANDOFF_PNG = EXPORT_DIR / "handoff_analysis.png"

plt.figure(figsize=(10, 4))
plt.plot(plot_df["timestamp"], plot_df["temperature_c"])
plt.axhline(30, linestyle="--")
plt.title("Temperature History with Alert Threshold")
plt.xlabel("Time")
plt.ylabel("Temperature (C)")
plt.tight_layout()
plt.savefig(TEMP_PNG, dpi=160)
plt.close()

plt.figure(figsize=(10, 4))
plt.plot(plot_df["timestamp"], plot_df["flow_lpm"])
plt.axhline(25, linestyle="--")
plt.title("Flow History with Low-Flow Threshold")
plt.xlabel("Time")
plt.ylabel("Flow (L/min)")
plt.tight_layout()
plt.savefig(FLOW_PNG, dpi=160)
plt.close()

plt.figure(figsize=(9, 4))
plt.bar(
    handoff_observation["step"].astype(str),
    handoff_observation["score"]
)
plt.ylim(0, 1.1)
plt.title("Silent Handoff Completion by Task")
plt.xlabel("Task")
plt.ylabel("Completion")
plt.tight_layout()
plt.savefig(HANDOFF_PNG, dpi=160)
plt.close()

print("Created report figures.")


# Part 15 — Generate a Professional Word Report

The report combines:

- deterministic KPI analysis;
- data-quality findings;
- operational incidents;
- dashboard figures;
- handoff performance;
- improvements;
- Ollama documentation review;
- Ollama operational review;
- Ollama handoff review;
- AI red-team findings;
- limitations and next steps.


In [ ]:
try:
    from docx import Document
    from docx.shared import Inches, Pt
except ModuleNotFoundError:
    %pip install -q python-docx
    from docx import Document
    from docx.shared import Inches, Pt

REPORT_FILE = EXPORT_DIR / "EGN321_Module8_Assignment8_1_Instructor_Analysis_Report.docx"

doc = Document()
doc.styles["Normal"].font.name = "Aptos"
doc.styles["Normal"].font.size = Pt(10.5)

doc.add_heading(
    "EGN 321 — Module 8 Assignment 8.1",
    0
)

doc.add_paragraph(
    "Instructor Demonstration Report: "
    "Silent Handoff, Operational Analysis, AI Review, and Improvement"
)

doc.add_heading("1. Executive Summary", level=1)

doc.add_paragraph(
    f"The demonstration analyzed {total_records} cooling-loop records. "
    f"{accepted_records} records were accepted and {rejected_records} were rejected, "
    f"for an acceptance rate of {acceptance_rate:.2f}%. "
    f"The system recorded {degraded_records} degraded-state rows, "
    f"{high_temp_rows} high-temperature alert rows, "
    f"{low_flow_rows} low-flow alert rows, and {large_gaps} large sampling gaps."
)

doc.add_paragraph(
    f"The Silent Handoff achieved {handoff_score:.1f}/{handoff_max} "
    f"({handoff_percent:.1f}%), corresponding to {handoff_level}."
)

doc.add_heading("2. Data Quality Summary", level=1)

table = doc.add_table(
    rows=1,
    cols=2
)

table.style = "Table Grid"

table.rows[0].cells[0].text = "Metric"
table.rows[0].cells[1].text = "Value"

for _, row in quality_summary.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["metric"])
    cells[1].text = str(row["value"])

doc.add_heading("3. Operational KPI Summary", level=1)

table = doc.add_table(
    rows=1,
    cols=2
)

table.style = "Table Grid"

table.rows[0].cells[0].text = "KPI"
table.rows[0].cells[1].text = "Value"

for _, row in kpis.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["KPI"])
    cells[1].text = str(row["Value"])

doc.add_heading("4. Temperature Analysis", level=1)
doc.add_picture(
    str(TEMP_PNG),
    width=Inches(6.5)
)

doc.add_heading("5. Flow Analysis", level=1)
doc.add_picture(
    str(FLOW_PNG),
    width=Inches(6.5)
)

doc.add_heading("6. Significant Operational Evidence", level=1)

doc.add_paragraph(
    "The following records were selected because they were rejected, "
    "degraded, part of an active alert, or associated with a large timing gap."
)

incident_table = doc.add_table(
    rows=1,
    cols=6
)

incident_table.style = "Table Grid"

headers = [
    "Record",
    "Timestamp",
    "Health",
    "Alert",
    "Reason",
    "Gap Sec"
]

for i, h in enumerate(headers):
    incident_table.rows[0].cells[i].text = h

for _, row in incident_records.head(20).iterrows():
    cells = incident_table.add_row().cells

    cells[0].text = str(row["record_id"])
    cells[1].text = str(row["timestamp"])
    cells[2].text = str(row["health_state"])
    cells[3].text = str(row["alert_state"])
    cells[4].text = str(row["reason"])
    cells[5].text = str(row["gap_seconds"])

doc.add_heading("7. Silent Handoff Performance", level=1)

doc.add_picture(
    str(HANDOFF_PNG),
    width=Inches(6.2)
)

handoff_table = doc.add_table(
    rows=1,
    cols=5
)

handoff_table.style = "Table Grid"

for i, h in enumerate([
    "Step",
    "Action",
    "Result",
    "Confusion",
    "Improvement"
]):
    handoff_table.rows[0].cells[i].text = h

for _, row in handoff_observation.iterrows():
    cells = handoff_table.add_row().cells
    cells[0].text = str(row["step"])
    cells[1].text = str(row["operator_action"])
    cells[2].text = str(row["success"])
    cells[3].text = str(row["confusion_observed"])
    cells[4].text = str(row["improvement_needed"])

doc.add_heading("8. Ollama Documentation Review", level=1)
doc.add_paragraph(ollama_documentation_review)

doc.add_heading("9. Ollama Operational Review", level=1)
doc.add_paragraph(ollama_operational_review)

doc.add_heading("10. Ollama Handoff Review", level=1)
doc.add_paragraph(ollama_handoff_review)

doc.add_heading("11. AI Red-Team Review", level=1)
doc.add_paragraph(ollama_red_team_review)

doc.add_heading("12. Instructor Conclusions", level=1)

doc.add_paragraph(
    "The application evidence demonstrates that handoff quality is not determined "
    "only by whether the software runs. A successful handoff requires clear setup, "
    "interpretable outputs, visible health information, accessible logs, actionable "
    "recovery steps, explicit verification, and documented limits."
)

doc.add_paragraph(
    "The AI reviews are supporting analysis only. Final operational claims remain "
    "grounded in the deterministic dataset, SQLite history, validation output, "
    "alert states, structured logs, and observed handoff evidence."
)

doc.save(REPORT_FILE)

print("Created:", REPORT_FILE)


# Part 16 — Build AI_LOG.md

The AI log makes AI usage transparent and auditable.


In [ ]:
AI_LOG_FILE = EXPORT_DIR / "AI_LOG.md"

AI_LOG_FILE.write_text(
f'''# AI_LOG.md

## Tool / Model
Ollama / {OLLAMA_MODEL}

## AI Use 1 — Documentation Audit
Purpose:
Review the operating guide from the perspective of a first-time operator.

### Output
{ollama_documentation_review}

---

## AI Use 2 — Operational / Anomaly Review
Purpose:
Review deterministic KPI and incident evidence.

### Deterministic Evidence Supplied
{json.dumps(analysis_summary, indent=2)}

### Output
{ollama_operational_review}

---

## AI Use 3 — Silent Handoff Review
Purpose:
Review human handoff evidence and identify friction/improvements.

### Output
{ollama_handoff_review}

---

## AI Use 4 — Red-Team Review
Purpose:
Challenge any AI claim stronger than the evidence.

### Output
{ollama_red_team_review}

---

## Verification Rule
AI output was not treated as operational evidence.

Final claims must be verified using:
- source records;
- validation rules;
- SQLite history;
- structured logs;
- deterministic Python calculations;
- observed handoff evidence.
''',
encoding="utf-8"
)

print("Created:", AI_LOG_FILE)


# Part 17 — Export Analysis Data and Reporting Files


In [ ]:
sensor_df.to_csv(
    EXPORT_DIR / "source_dataset_900_records.csv",
    index=False
)

history.to_csv(
    EXPORT_DIR / "validated_history_900_records.csv",
    index=False
)

incident_records.to_csv(
    EXPORT_DIR / "incident_records.csv",
    index=False
)

quality_summary.to_csv(
    EXPORT_DIR / "data_quality_summary.csv",
    index=False
)

kpis.to_csv(
    EXPORT_DIR / "operational_kpis.csv",
    index=False
)

handoff_observation.to_csv(
    EXPORT_DIR / "handoff_observation.csv",
    index=False
)

(EXPORT_DIR / "analysis_summary.json").write_text(
    json.dumps(
        analysis_summary,
        indent=2
    ),
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_documentation_review.txt").write_text(
    ollama_documentation_review,
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_operational_review.txt").write_text(
    ollama_operational_review,
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_handoff_review.txt").write_text(
    ollama_handoff_review,
    encoding="utf-8"
)

(EXPORT_DIR / "ollama_red_team_review.txt").write_text(
    ollama_red_team_review,
    encoding="utf-8"
)

shutil.copy2(
    DB_FILE,
    EXPORT_DIR / "cooling_loop.db"
)

shutil.copy2(
    LOG_FILE,
    EXPORT_DIR / "system_log.jsonl"
)

print("Analysis files exported.")


# Part 18 — Final Submission / Instructor Evidence Package


In [ ]:
MANIFEST = EXPORT_DIR / "SUBMISSION_MANIFEST.txt"

MANIFEST.write_text(
'''MODULE 8 ASSIGNMENT 8.1 — INSTRUCTOR DEMO PACKAGE

1. Professional Word analysis report
2. Interactive HTML dashboard
3. 900-record source dataset
4. Validated application history
5. Incident records
6. Data-quality summary
7. Operational KPI table
8. SQLite database
9. JSONL system log
10. Silent handoff observation
11. AI_LOG.md
12. Ollama documentation review
13. Ollama operational review
14. Ollama handoff review
15. Ollama red-team review
16. Static report figures

COURSE RULE:
Deterministic evidence is authoritative.
AI supports analysis, communication, and improvement.
''',
encoding="utf-8"
)

ZIP_FILE = Path(
    "/content/EGN321_Module8_Assignment8_1_COMPLETE_ANALYSIS_DEMO.zip"
)

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in sorted(
        EXPORT_DIR.iterdir()
    ):

        if p.is_file():
            z.write(
                p,
                arcname=p.name
            )

print("✅ Complete demo ZIP:", ZIP_FILE)

print("\nFiles:")
for p in sorted(EXPORT_DIR.iterdir()):
    print(" -", p.name)

# Optional automatic download:
# from google.colab import files
# files.download(str(ZIP_FILE))


# Instructor Closing Discussion

Use the finished dashboard and report to ask:

> **What can we prove from the data?**

> **What is only a hypothesis?**

> **What did Ollama add that Python did not?**

> **What did Python/SQLite/logs prove that Ollama could not?**

> **Could a new operator understand both the application and the analysis without the developer present?**

The final lesson is:

**Data creates evidence.  
Dashboards make evidence visible.  
AI helps interpret and challenge.  
Reporting makes the evidence transferable.  
Handoff proves whether the system is truly usable.**
